# 02_pySCENIC.ipynb

**Purpose:** Run the complete pySCENIC workflow to infer transcription factor regulatory networks from scRNA-seq data.

**Input:**
- `data/gallitano_loom.loom` — loom file generated from Seurat object via `01_preprocessing.R`
- `data/allTFs_mm.txt` — mouse TF list from https://resources.aertslab.org/cistarget/tf_lists/

**Output:**
- `data/gallitano_fixed.loom` — loom file with corrected structure for pySCENIC
- `output/adj.csv` — GRN adjacency matrix (~62MB, ~1.97M TF-gene relationships)
- `output/regulons.csv` — pruned regulons from cisTarget step
- `output/auc_mtx.loom` — AUCell regulon activity scores per cell (23,097 cells x 130 regulons)

**Workflow:**
1. Download mm10 cisTarget databases and motif annotations
2. Fix loom file structure from R conversion
3. GRN inference (`pyscenic grn`) — ~6.5 hours on 4 CPUs
4. Regulon pruning via cisTarget (`pyscenic ctx`)
5. AUCell scoring (`pyscenic aucell`) — ~4 minutes on 1 CPU

**Notes:**
- Run on Kaggle (free CPU compute) due to memory and runtime requirements
- pySCENIC 0.12.1 requires specific dask/numpy versions — see compatibility cells
- Output `auc_mtx.loom` is imported into R via `SeuratExtend::ImportPyscenicLoom()` in `02b_pySCENIC_TFAnalysis.R`
- Dataset: 23,013 genes x 23,097 cells across 7 brain cell types
- Conditions: WT_SD (sleep deprived) vs WT_SD_Ctrl (control)

**Author:** Karthikeya Kodali

In [ ]:
# ── 1. Create database directory ──────────────────────────────────────────────
import os
os.makedirs('databases', exist_ok=True)

In [ ]:
# ── 2. Download mm10 cisTarget ranking databases and motif annotations ────────
# Mouse mm10, RefSeq r80, mc_v10 clustering
# 10kbp window: ~226MB | 500bp window: ~227MB | motifs: ~108MB
# Source: https://resources.aertslab.org/cistarget/

!wget https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
    -O databases/mm10_10kbp.feather

!wget https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
    -O databases/mm10_500bp.feather

!wget https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl \
    -O databases/motifs.tbl

In [ ]:
# ── 3. Rename feather files to full pySCENIC expected filenames ───────────────
# pySCENIC ctx expects full filename when passed as positional arguments
# Verify download sizes before renaming

!ls -lh databases/

!mv databases/mm10_10kbp.feather \
    databases/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather

!mv databases/mm10_500bp.feather \
    databases/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather

In [ ]:
# ── 4. Install scanpy and set TF list path ────────────────────────────────────
# scanpy is required for loom file handling
# TF list must be uploaded separately to Kaggle as a dataset
# Download from: https://resources.aertslab.org/cistarget/tf_lists/allTFs_mm.txt

!pip install scanpy
import scanpy as sc

# Kaggle input dataset path for mouse TF list
f_tfs = "/kaggle/input/mouse-tfs/allTFs_mm.txt"

In [ ]:
# ── 5. Install pySCENIC ───────────────────────────────────────────────────────
# Uncomment second line to install from GitHub (more recent but less stable)

!pip install pyscenic
#!pip install git+https://github.com/aertslab/pySCENIC.git

In [ ]:
# ── 6. pySCENIC compatibility fixes ──────────────────────────────────────────
# pySCENIC 0.12.1 conflicts with recent dask and numpy on Kaggle
# These pinned versions are required for stable execution
# Dependency conflict warnings in the output can be safely ignored

!pip install dask-expr==0.5.3 distributed==2024.2.1
!pip install 'numpy>=1.23,<1.24'

In [ ]:
# ── 7. Define file paths and import libraries ─────────────────────────────────
import os
import pyscenic
import pandas as pd
import loompy as lp
import numpy as np

# Input loom file (uploaded as Kaggle dataset)
loom_input = "/kaggle/input/gallitano-loom-final/gallitano_loom.loom"

# cisTarget database files
db_fnames = [
    'databases/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather',
    'databases/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
]
motif_annotations = 'databases/motifs.tbl'

In [ ]:
# ── 8. Fix loom file structure from R conversion ──────────────────────────────
# SeuratExtend::Seu2Loom() produces a loom structure that pySCENIC cannot
# process directly — read and rewrite with loompy to correct the format
# Expected output: 23,013 genes x 23,097 cells

with lp.connect(loom_input, mode='r', validate=False) as ds_in:
    matrix    = ds_in[:, :]
    row_attrs = {key: ds_in.ra[key][:] for key in ds_in.ra.keys()}
    col_attrs = {key: ds_in.ca[key][:] for key in ds_in.ca.keys()}

print(f"Read {matrix.shape[0]} genes x {matrix.shape[1]} cells")

lp.create('gallitano_fixed.loom', matrix, row_attrs, col_attrs)
print("Fixed loom created: gallitano_fixed.loom")

In [ ]:
# ── 9. GRN inference ──────────────────────────────────────────────────────────
# Infer gene regulatory network using GRNBoost2
# Runtime: ~6.5 hours on 4 CPUs for this dataset
# Output: adj.csv (~62MB, ~1.97M TF-gene adjacencies)
# Adjust --num_workers to match available CPU cores

!pyscenic grn gallitano_fixed.loom {f_tfs} \
    -o adj.csv \
    --num_workers 4 \
    --seed 123

In [ ]:
# ── 9a. Sanity check adjacency file ──────────────────────────────────────────
# Expected: ~62MB, ~1.97M lines
# Columns: TF, target, importance

if os.path.exists('adj.csv'):
    size_mb = os.path.getsize('adj.csv') / (1024*1024)
    print(f"adj.csv size: {size_mb:.2f} MB")
    with open('adj.csv', 'r') as f:
        lines = sum(1 for line in f)
    print(f"Total adjacencies: {lines:,}")
else:
    print("No output file found — GRN step may still be running")

adjacencies = pd.read_csv('adj.csv')
adjacencies.head()

In [ ]:
# ── 10. Regulon pruning via cisTarget ─────────────────────────────────────────
# Prune co-expression modules to TF regulons using motif enrichment
# against mm10 cisTarget databases
# Adjust --num_workers to match available CPU cores

!pyscenic ctx \
    adj.csv \
    databases/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
    databases/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather \
    --annotations_fname databases/motifs.tbl \
    --expression_mtx_fname gallitano_fixed.loom \
    --output regulons.csv \
    --num_workers 4

In [ ]:
# ── 11. Pandas compatibility fix for AUCell loom output ───────────────────────
# Generating .loom output from AUCell requires pandas < 2.0
# Must run this before the aucell step
# Skip if outputting .csv instead

!pip install "pandas<2.0"

In [ ]:
# ── 12. AUCell scoring ────────────────────────────────────────────────────────
# Calculate AUCell regulon activity scores for each cell
# Runtime: ~4 minutes on 1 CPU
# Output: auc_mtx.loom (23,097 cells x 130 regulons)
# Change --output extension to .csv if loom output fails due to pandas version
# Adjust --num_workers to match available CPU cores

!pyscenic aucell \
    gallitano_fixed.loom \
    regulons.csv \
    --output auc_mtx.loom \
    --num_workers 1

In [ ]:
# ── 12a. Verify AUCell output (only if .csv output was used) ──────────────────
# Skip this cell if .loom output was used — import directly into R
# Expected shape: 23,097 cells x 130 regulons

auc = pd.read_csv("auc_mtx.csv", index_col=0)
print(f"AUC matrix shape: {auc.shape} (cells x regulons)")
print(auc.head())

## Workflow complete

Primary output is `auc_mtx.loom` containing AUCell regulon activity scores for 23,097 cells across 130 inferred TF regulons.

Import into R for downstream analysis:
```r
data <- ImportPyscenicLoom("data/auc_mtx.loom", seu = data)
```
See `05_pySCENIC_analysis.R` for full downstream visualization and differential TF activity analysis.